# Experiments 37-40 — Quarters and Eighths vs. 35Vector / 19Vector

Renumbering note: your "29-32" collides twice over with earlier work
(29-30 were HizbPair vs 35Vector/19Vector; 31-32 belonged to an earlier,
unrelated experiment track). This is **Experiments 37-40**, continuing the
clustering line from where it left off at 36.

- **37:** 240 Quranic quarters vs. 35Vector
- **38:** 240 Quranic quarters vs. 19Vector
- **39:** 480 Quranic eighths vs. 35Vector
- **40:** 480 Quranic eighths vs. 19Vector

**What "quarters" and "eighths" mean here:** the Quran's own API
metadata already divides it into 240 "hizb quarters" (rub' al-hizb) —
that's the finest official traditional division, four per hizb,
already used elsewhere in this project. "Eighths" aren't an official
division; each of those 240 quarters is split in half by verse order
(first half of that quarter's ayat = one eighth, second half = the
other), giving 480 total. If a quarter has an odd number of ayat, the
first half gets the extra one — stated here so the rule is transparent,
not hidden in code.

Same pattern as Experiments 25-28: each poet vector is a single point
introduced among many separate Quranic units, so the same "lone point"
caution applies. Full cluster tables, ranked similarity breakdowns, and
graphs for all four, same as the Umayya notebook.

**Reuses all your cached embeddings** — no new model computation
needed, should run fast.

**Before you start:** put `poems.db` in the same folder as this
notebook.

Run cells top to bottom, **Shift+Enter**.

In [1]:
# CELL 1 -- Install packages
!pip -q install sentence-transformers torch scikit-learn umap-learn hdbscan pandas numpy matplotlib seaborn scipy requests


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# CELL 2 -- Configuration
import re, sqlite3, hashlib, json, warnings, pickle
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

warnings.filterwarnings("ignore")

DB_PATH = Path("poems.db")
CACHE_DIR = Path("quran_cache"); CACHE_DIR.mkdir(exist_ok=True)
EMBED_CACHE_DIR = Path("embed_cache"); EMBED_CACHE_DIR.mkdir(exist_ok=True)
FIGURES_DIR = Path("output/figures"); FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR = Path("output/tables"); TABLES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = Path("output/reports"); REPORTS_DIR.mkdir(parents=True, exist_ok=True)

SBERT_MODEL_NAME = "akhooli/Arabic-SBERT-100K"
MAX_VERSES_PER_POEM = 20

UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1
UMAP_N_COMPONENTS_HIGH = 50
UMAP_METRIC = "cosine"
HDBSCAN_MIN_CLUSTER_SIZE = 5
HDBSCAN_MIN_SAMPLES = 3

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
import torch
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    print("GPU available:", torch.cuda.get_device_name(0))
else:
    print("No GPU found -- will run on CPU.")

def split_verses(poem_text):
    if not poem_text or not isinstance(poem_text, str):
        return []
    verses = re.split(r'[\n\r]+|[.!\u061F?\u061B;]+', poem_text)
    return [v.strip() for v in verses if len(v.strip()) > 10]

def normalize_word(w):
    w = re.sub(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06ED\u0670]", "", w)
    w = re.sub(r"\u0640", "", w)
    w = re.sub(r"[\u0622\u0623\u0625\u0671]", "\u0627", w)
    return w.strip()

plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 300, "savefig.bbox": "tight", "font.size": 11})
print("Config loaded.")

GPU available: NVIDIA GeForce RTX 4080 SUPER
Config loaded.


In [3]:
# CELL 3 -- Load the 260 poets from poems.db
conn = sqlite3.connect(str(DB_PATH))
df = pd.read_sql_query(
    "SELECT poet_name, poem_title, poem_text, poem_type, poem_meter, verses_count "
    "FROM poems WHERE poet_name IS NOT NULL AND poem_text IS NOT NULL "
    "AND LENGTH(poem_text) >= 50",
    conn,
)
conn.close()

df["poem_hash"] = df["poem_text"].apply(lambda t: hashlib.md5(t.strip().encode("utf-8")).hexdigest())
df = df.drop_duplicates(subset=["poem_hash"]).copy()

poems_by_poet = defaultdict(list)
for _, row in df.iterrows():
    poems_by_poet[row["poet_name"]].append(row["poem_text"])
poems_by_poet = dict(poems_by_poet)

poet_total_verses = {
    poet: sum(len(split_verses(p)) for p in poems)
    for poet, poems in poems_by_poet.items()
}

print(f"Poets loaded: {len(poems_by_poet)}")

DatabaseError: Execution failed on sql 'SELECT poet_name, poem_title, poem_text, poem_type, poem_meter, verses_count FROM poems WHERE poet_name IS NOT NULL AND poem_text IS NOT NULL AND LENGTH(poem_text) >= 50': no such table: poems

In [ ]:
# CELL 4 -- Fetch Quran text WITH hizb-quarter metadata
cache_file = CACHE_DIR / "quran_ayat_with_hizb.json"

if cache_file.exists():
    print("Loading Quran (with hizb metadata) from local cache...")
    with open(cache_file, encoding="utf-8") as f:
        surahs_raw = json.load(f)
else:
    print("Fetching Quran from Al Quran Cloud API...")
    resp = requests.get("https://api.alquran.cloud/v1/quran/quran-uthmani", timeout=60)
    resp.raise_for_status()
    surahs_raw = resp.json()["data"]["surahs"]
    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(surahs_raw, f, ensure_ascii=False)
    print("Fetched and cached.")

surah_names = {}
surah_poem_text = {}
ayah_records = []
for s in surahs_raw:
    snum = s["number"]
    surah_names[snum] = s["englishName"]
    ayat_texts = [a["text"] for a in s["ayahs"]]
    surah_poem_text[snum] = "\n".join(ayat_texts)
    for idx, a in enumerate(s["ayahs"]):
        hizb_number = ((a["hizbQuarter"] - 1) // 4) + 1
        ayah_records.append({
            "surah_number": snum, "ayah_number": a["numberInSurah"],
            "text": a["text"], "hizb_number": hizb_number,
            "hizb_quarter": a["hizbQuarter"], "global_order": None,  # filled below
        })

ayah_df = pd.DataFrame(ayah_records)
ayah_df["global_order"] = range(len(ayah_df))  # preserves true Quranic reading order
print(f"Surahs: {ayah_df['surah_number'].nunique()} | Ayat: {len(ayah_df)} | "
      f"Hizb quarters found: {ayah_df['hizb_quarter'].nunique()} (should be 240)")

In [ ]:
# CELL 5 -- Load poet embeddings (cached), build 35Vector
from sentence_transformers import SentenceTransformer

poet_cache_file = EMBED_CACHE_DIR / "poet_embeddings.pkl"

print("Loading model:", SBERT_MODEL_NAME)
model = SentenceTransformer(SBERT_MODEL_NAME)
print("Loaded.")

def _encode(texts, batch_size=64):
    if not texts:
        return np.array([])
    return model.encode(texts, batch_size=batch_size, show_progress_bar=False,
                         normalize_embeddings=True, convert_to_numpy=True)

def embed_poem_verse_average(poems_by_author, max_verses=MAX_VERSES_PER_POEM, seed=RANDOM_SEED):
    out = {}
    rng = np.random.RandomState(seed)
    for author, poems in poems_by_author.items():
        poem_vectors = []
        for poem in poems:
            verses = split_verses(poem)
            if len(verses) > max_verses:
                idx = rng.choice(len(verses), max_verses, replace=False)
                verses = [verses[i] for i in sorted(idx)]
            if verses:
                poem_vectors.append(np.mean(_encode(verses), axis=0))
        if poem_vectors:
            out[author] = np.mean(poem_vectors, axis=0)
    return out

if poet_cache_file.exists():
    print("Loading cached poet embeddings...")
    with open(poet_cache_file, "rb") as f:
        poet_embeddings = pickle.load(f)
    print(f"Loaded {len(poet_embeddings)} cached poet embeddings.")
else:
    print("Embedding 260 poets (slow, one-time only)...")
    poet_embeddings = embed_poem_verse_average(poems_by_poet)
    with open(poet_cache_file, "wb") as f:
        pickle.dump(poet_embeddings, f)
    print(f"Done and cached. {len(poet_embeddings)} poets embedded.")

max_surah_verses = max(len(split_verses(t)) for t in surah_poem_text.values())
poets_35 = [p for p, v in poet_total_verses.items() if v > max_surah_verses]
poets_35_embeddings = {p: poet_embeddings[p] for p in poets_35 if p in poet_embeddings}
vector_35 = np.mean(list(poets_35_embeddings.values()), axis=0)
print(f"\n35Vector built from {len(poets_35_embeddings)} well-documented poets.")

In [ ]:
# CELL 6 -- Load ayah embeddings (cached), build quarter/eighth/whole vectors
ayah_embed_cache_file = EMBED_CACHE_DIR / "ayah_embeddings.pkl"

if ayah_embed_cache_file.exists():
    print("Loading cached ayah embeddings...")
    with open(ayah_embed_cache_file, "rb") as f:
        ayah_embeddings = pickle.load(f)
    print(f"Loaded {len(ayah_embeddings)} cached ayah embeddings.")
else:
    print(f"Embedding all {len(ayah_df)} ayat individually (slow, one-time only)...")
    all_texts = ayah_df["text"].tolist()
    all_vecs = _encode(all_texts, batch_size=64)
    ayah_embeddings = {}
    for (snum, anum), vec in zip(zip(ayah_df["surah_number"], ayah_df["ayah_number"]), all_vecs):
        ayah_embeddings[(snum, anum)] = vec
    with open(ayah_embed_cache_file, "wb") as f:
        pickle.dump(ayah_embeddings, f)
    print(f"Done and cached. {len(ayah_embeddings)} ayat embedded.")

def vector_for_ayat(ayat_keys):
    vecs = [ayah_embeddings[k] for k in ayat_keys if k in ayah_embeddings]
    return np.mean(vecs, axis=0) if vecs else None

# --- 240 quarters: group directly by the hizb_quarter field ---
quarter_vectors = {}
for qnum, group in ayah_df.groupby("hizb_quarter"):
    keys = list(zip(group["surah_number"], group["ayah_number"]))
    v = vector_for_ayat(keys)
    if v is not None:
        quarter_vectors[qnum] = v
print(f"Quarter vectors built: {len(quarter_vectors)} (should be 240)")

# --- 480 eighths: split each quarter's ayat in half by reading order ---
eighth_vectors = {}
for qnum, group in ayah_df.groupby("hizb_quarter"):
    group_sorted = group.sort_values("global_order")
    n = len(group_sorted)
    half = (n + 1) // 2  # first half gets the extra ayah if odd
    first_half = group_sorted.iloc[:half]
    second_half = group_sorted.iloc[half:]
    keys_a = list(zip(first_half["surah_number"], first_half["ayah_number"]))
    keys_b = list(zip(second_half["surah_number"], second_half["ayah_number"]))
    v_a = vector_for_ayat(keys_a)
    v_b = vector_for_ayat(keys_b)
    if v_a is not None:
        eighth_vectors[f"{qnum}a"] = v_a
    if v_b is not None:
        eighth_vectors[f"{qnum}b"] = v_b
print(f"Eighth vectors built: {len(eighth_vectors)} (should be 480)")

# --- Whole Quran vector (needed to re-derive 19Vector) ---
surah_vectors = {}
for snum in surah_names:
    keys = list(zip(ayah_df[ayah_df["surah_number"] == snum]["surah_number"],
                    ayah_df[ayah_df["surah_number"] == snum]["ayah_number"]))
    v = vector_for_ayat(keys)
    if v is not None:
        surah_vectors[snum] = v
quran_whole_vector = np.mean(list(surah_vectors.values()), axis=0)
print(f"Surah vectors (for 19Vector derivation): {len(surah_vectors)}")

In [ ]:
# CELL 7 -- Clustering helper, then derive 19Vector (rerun of Experiment 1's setup)
import umap, hdbscan
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score

def run_umap(matrix, n_components, n_neighbors, min_dist, metric="cosine", random_state=RANDOM_SEED):
    reducer = umap.UMAP(n_neighbors=min(n_neighbors, len(matrix) - 1), n_components=n_components,
                         min_dist=min_dist, metric=metric, random_state=random_state)
    return reducer.fit_transform(matrix)

def cluster_and_report(embeddings_dict, label_types, title, n_neighbors=UMAP_N_NEIGHBORS,
                       min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE, verbose=True):
    names = sorted(embeddings_dict.keys())
    n = len(names)
    emb_matrix = np.array([embeddings_dict[nm] for nm in names])
    umap_high = run_umap(emb_matrix, min(UMAP_N_COMPONENTS_HIGH, max(2, n - 2)), n_neighbors, 0.0, UMAP_METRIC)
    clusterer = hdbscan.HDBSCAN(min_cluster_size=min(min_cluster_size, max(2, n // 10)),
                                min_samples=HDBSCAN_MIN_SAMPLES,
                                cluster_selection_method="eom", metric="euclidean")
    labels = clusterer.fit_predict(umap_high)
    umap_2d = run_umap(emb_matrix, 2, n_neighbors, UMAP_MIN_DIST, UMAP_METRIC)
    mask = labels >= 0
    n_clusters = len(set(labels[mask])) if mask.sum() > 0 else 0
    n_outliers = int(np.sum(labels == -1))
    sil = float(silhouette_score(umap_high[mask], labels[mask])) if n_clusters >= 2 and mask.sum() > n_clusters else None
    result_df = pd.DataFrame({"name": names, "type": [label_types[nm] for nm in names],
        "cluster": labels, "umap_x": umap_2d[:, 0], "umap_y": umap_2d[:, 1]})
    if verbose:
        print(f"=== {title} ===")
        print(f"Total: {n} | Clusters: {n_clusters} | Outliers: {n_outliers} | Silhouette: {sil}")
    return result_df, {"n_entities": n, "n_clusters": n_clusters, "n_outliers": n_outliers, "silhouette": sil}

print("Re-deriving 19Vector by rerunning Experiment 1 (all 260 poets + Quran whole)...")
exp1_rerun_embeddings = dict(poet_embeddings)
exp1_rerun_embeddings["Quran (whole)"] = quran_whole_vector
names_1 = sorted(exp1_rerun_embeddings.keys())
n_1 = len(names_1)
emb_matrix_1 = np.array([exp1_rerun_embeddings[nm] for nm in names_1])
umap_high_1 = run_umap(emb_matrix_1, min(UMAP_N_COMPONENTS_HIGH, n_1 - 2), UMAP_N_NEIGHBORS, 0.0)
clusterer_1 = hdbscan.HDBSCAN(min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE, min_samples=HDBSCAN_MIN_SAMPLES,
                              cluster_selection_method="eom", metric="euclidean")
labels_1 = clusterer_1.fit_predict(umap_high_1)
labels_1_by_name = dict(zip(names_1, labels_1))
quran_cluster_1 = labels_1_by_name["Quran (whole)"]
poets_19_names = [nm for nm in names_1 if nm != "Quran (whole)" and labels_1_by_name[nm] == quran_cluster_1]
poets_19_embeddings = {p: poet_embeddings[p] for p in poets_19_names}
vector_19 = np.mean(list(poets_19_embeddings.values()), axis=0) if poets_19_embeddings else None
print(f"19Vector built from {len(poets_19_embeddings)} poets sharing the Quran's cluster in this rerun")
print("(small variation from the original 19 is normal run-to-run UMAP/HDBSCAN noise).")

## Experiments 37-38 — 240 Quarters vs. 35Vector and 19Vector

In [ ]:
# CELL 8 -- Experiments 37 and 38: 240 quarters vs 35Vector / 19Vector
def cluster_lone_vector(vector_dict_entry, vector_label, units_dict, unit_label, exp_num):
    embeddings = {vector_label: vector_dict_entry, **units_dict}
    types = {nm: (vector_label if nm == vector_label else unit_label) for nm in embeddings}
    result_df, metrics = cluster_and_report(embeddings, types, f"Experiment {exp_num}: {vector_label} vs {unit_label}")
    vec_row = result_df[result_df["name"] == vector_label].iloc[0]
    status = "OUTLIER" if vec_row["cluster"] == -1 else f"Cluster {vec_row['cluster']}"
    same_cluster = result_df[(result_df["cluster"] == vec_row["cluster"]) & (result_df["name"] != vector_label)]
    print(f"{vector_label} landed in: {status}")
    print(f"Units sharing that cluster: {len(same_cluster)}")
    return result_df, metrics

def similarity_breakdown(vector_entry, units_dict):
    names = sorted(units_dict.keys())
    vecs = np.array([units_dict[n] for n in names])
    sims = cosine_similarity([vector_entry], vecs)[0]
    df = pd.DataFrame({"name": names, "cosine_similarity": sims}).sort_values(
        "cosine_similarity", ascending=False).reset_index(drop=True)
    return df

quarter_units = {f"Quarter {n}": v for n, v in quarter_vectors.items()}

# --- Experiment 37: 240 quarters vs 35Vector ---
exp37_df, exp37_metrics = cluster_lone_vector(vector_35, "35Vector", quarter_units, "Quarter", 37)
exp37_df.to_csv(TABLES_DIR / "experiment37_clusters.csv", index=False)
exp37_sim = similarity_breakdown(vector_35, quarter_units)
exp37_sim.to_csv(TABLES_DIR / "experiment37_similarity_breakdown.csv", index=False)
print(f"\nTop 10 most similar quarters to 35Vector:")
print(exp37_sim.head(10).to_string(index=False))
print(f"Similarity stats: mean={exp37_sim['cosine_similarity'].mean():.4f}, median={exp37_sim['cosine_similarity'].median():.4f}")

# --- Experiment 38: 240 quarters vs 19Vector ---
print("\n" + "=" * 70)
if vector_19 is not None:
    exp38_df, exp38_metrics = cluster_lone_vector(vector_19, "19Vector", quarter_units, "Quarter", 38)
    exp38_df.to_csv(TABLES_DIR / "experiment38_clusters.csv", index=False)
    exp38_sim = similarity_breakdown(vector_19, quarter_units)
    exp38_sim.to_csv(TABLES_DIR / "experiment38_similarity_breakdown.csv", index=False)
    print(f"\nTop 10 most similar quarters to 19Vector:")
    print(exp38_sim.head(10).to_string(index=False))
    print(f"Similarity stats: mean={exp38_sim['cosine_similarity'].mean():.4f}, median={exp38_sim['cosine_similarity'].median():.4f}")
else:
    print("Experiment 38 skipped -- 19Vector could not be derived this run.")
    exp38_metrics = None

## Experiments 39-40 — 480 Eighths vs. 35Vector and 19Vector

In [ ]:
# CELL 9 -- Experiments 39 and 40: 480 eighths vs 35Vector / 19Vector
eighth_units = {f"Eighth {n}": v for n, v in eighth_vectors.items()}

# --- Experiment 39: 480 eighths vs 35Vector ---
exp39_df, exp39_metrics = cluster_lone_vector(vector_35, "35Vector", eighth_units, "Eighth", 39)
exp39_df.to_csv(TABLES_DIR / "experiment39_clusters.csv", index=False)
exp39_sim = similarity_breakdown(vector_35, eighth_units)
exp39_sim.to_csv(TABLES_DIR / "experiment39_similarity_breakdown.csv", index=False)
print(f"\nTop 10 most similar eighths to 35Vector:")
print(exp39_sim.head(10).to_string(index=False))
print(f"Similarity stats: mean={exp39_sim['cosine_similarity'].mean():.4f}, median={exp39_sim['cosine_similarity'].median():.4f}")

# --- Experiment 40: 480 eighths vs 19Vector ---
print("\n" + "=" * 70)
if vector_19 is not None:
    exp40_df, exp40_metrics = cluster_lone_vector(vector_19, "19Vector", eighth_units, "Eighth", 40)
    exp40_df.to_csv(TABLES_DIR / "experiment40_clusters.csv", index=False)
    exp40_sim = similarity_breakdown(vector_19, eighth_units)
    exp40_sim.to_csv(TABLES_DIR / "experiment40_similarity_breakdown.csv", index=False)
    print(f"\nTop 10 most similar eighths to 19Vector:")
    print(exp40_sim.head(10).to_string(index=False))
    print(f"Similarity stats: mean={exp40_sim['cosine_similarity'].mean():.4f}, median={exp40_sim['cosine_similarity'].median():.4f}")
else:
    print("Experiment 40 skipped -- 19Vector could not be derived this run.")
    exp40_metrics = None

In [ ]:
# CELL 10 -- Figures
def plot_vs_vector(df, title, save_name, vector_label, unit_marker):
    fig, ax = plt.subplots(figsize=(12, 9))
    unique_clusters = sorted(df["cluster"].unique())
    n_clust_plot = len([c for c in unique_clusters if c >= 0])
    colors = plt.cm.tab20(np.linspace(0, 1, max(n_clust_plot, 1)))
    for cl in unique_clusters:
        sub = df[df["cluster"] == cl]
        color = "gray" if cl == -1 else colors[cl % len(colors)]
        units = sub[sub["name"] != vector_label]
        vec = sub[sub["name"] == vector_label]
        if len(units) > 0:
            ax.scatter(units["umap_x"], units["umap_y"], c=[color], marker=unit_marker, s=60,
                      alpha=0.75, edgecolors="black", linewidth=0.3)
        if len(vec) > 0:
            ax.scatter(vec["umap_x"], vec["umap_y"], c="red", marker="*", s=550,
                      edgecolors="black", linewidth=1.5, zorder=10, label=vector_label)
    ax.set_xlabel("UMAP Dimension 1"); ax.set_ylabel("UMAP Dimension 2")
    ax.set_title(title)
    ax.legend(loc="best")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / save_name, dpi=300, bbox_inches="tight")
    plt.show()

plot_vs_vector(exp37_df, "Experiment 37: 240 Quarters vs 35Vector", "experiment37_umap.png", "35Vector", "^")
if exp38_metrics is not None:
    plot_vs_vector(exp38_df, "Experiment 38: 240 Quarters vs 19Vector", "experiment38_umap.png", "19Vector", "^")
plot_vs_vector(exp39_df, "Experiment 39: 480 Eighths vs 35Vector", "experiment39_umap.png", "35Vector", "v")
if exp40_metrics is not None:
    plot_vs_vector(exp40_df, "Experiment 40: 480 Eighths vs 19Vector", "experiment40_umap.png", "19Vector", "v")

# Similarity distribution histograms, all four together
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
plot_data = [(axes[0], exp37_sim, "37: Quarters vs 35Vector")]
if exp38_metrics is not None:
    plot_data.append((axes[1], exp38_sim, "38: Quarters vs 19Vector"))
plot_data.append((axes[2], exp39_sim, "39: Eighths vs 35Vector"))
if exp40_metrics is not None:
    plot_data.append((axes[3], exp40_sim, "40: Eighths vs 19Vector"))
for ax, sim_df, title in plot_data:
    ax.hist(sim_df["cosine_similarity"], bins=20, color="#3498db", edgecolor="black")
    ax.axvline(sim_df["cosine_similarity"].mean(), color="red", linestyle="--",
              label=f"mean={sim_df['cosine_similarity'].mean():.3f}")
    ax.set_xlabel("Cosine similarity"); ax.set_title(title)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "experiment37_38_39_40_similarity_distributions.png", dpi=300, bbox_inches="tight")
plt.show()

print("All figures saved.")

In [ ]:
# CELL 11 -- Final report
report_path = REPORTS_DIR / "experiments_37_40_report.txt"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("=" * 70 + "\n")
    f.write("EXPERIMENTS 37-40: QUARTERS AND EIGHTHS VS 35VECTOR / 19VECTOR\n")
    f.write("=" * 70 + "\n\n")

    f.write("EXPERIMENT 37: 240 QUARTERS VS 35VECTOR\n" + "-" * 40 + "\n")
    for k, v in exp37_metrics.items():
        f.write(f"  {k}: {v}\n")
    f.write("  Top 10 most similar quarters:\n")
    f.write(exp37_sim.head(10).to_string(index=False) + "\n")
    f.write(f"  Similarity stats: mean={exp37_sim['cosine_similarity'].mean():.4f}, "
            f"median={exp37_sim['cosine_similarity'].median():.4f}\n")

    if exp38_metrics is not None:
        f.write("\nEXPERIMENT 38: 240 QUARTERS VS 19VECTOR\n" + "-" * 40 + "\n")
        for k, v in exp38_metrics.items():
            f.write(f"  {k}: {v}\n")
        f.write("  Top 10 most similar quarters:\n")
        f.write(exp38_sim.head(10).to_string(index=False) + "\n")
        f.write(f"  Similarity stats: mean={exp38_sim['cosine_similarity'].mean():.4f}, "
                f"median={exp38_sim['cosine_similarity'].median():.4f}\n")

    f.write("\nEXPERIMENT 39: 480 EIGHTHS VS 35VECTOR\n" + "-" * 40 + "\n")
    for k, v in exp39_metrics.items():
        f.write(f"  {k}: {v}\n")
    f.write("  Top 10 most similar eighths:\n")
    f.write(exp39_sim.head(10).to_string(index=False) + "\n")
    f.write(f"  Similarity stats: mean={exp39_sim['cosine_similarity'].mean():.4f}, "
            f"median={exp39_sim['cosine_similarity'].median():.4f}\n")

    if exp40_metrics is not None:
        f.write("\nEXPERIMENT 40: 480 EIGHTHS VS 19VECTOR\n" + "-" * 40 + "\n")
        for k, v in exp40_metrics.items():
            f.write(f"  {k}: {v}\n")
        f.write("  Top 10 most similar eighths:\n")
        f.write(exp40_sim.head(10).to_string(index=False) + "\n")
        f.write(f"  Similarity stats: mean={exp40_sim['cosine_similarity'].mean():.4f}, "
                f"median={exp40_sim['cosine_similarity'].median():.4f}\n")

    f.write("\nHOW TO READ THIS\n" + "-" * 40 + "\n")
    f.write("  Same 'lone point' caution as Experiments 19-21 and 25-28: 35Vector\n")
    f.write("  and 19Vector are each a single point facing many Quranic units, so\n")
    f.write("  cluster membership alone tends toward absorption regardless of\n")
    f.write("  content. The similarity breakdown numbers are the more informative\n")
    f.write("  part of this result.\n")

print(f"Report written to {report_path}")
print()
print(open(report_path, encoding="utf-8").read())

## Done

Output in `output/`:
- `output/tables/experiment37/38/39/40_clusters.csv` — cluster membership
- `output/tables/experiment37/38/39/40_similarity_breakdown.csv` — full
  ranked similarity to every single quarter/eighth
- `output/figures/` — UMAP plots for all four, plus a combined
  similarity-distribution histogram
- `output/reports/experiments_37_40_report.txt` — everything together

Send this back and I'll build the report.